<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# FFNN Series de tiempo. - Predicción de eventos en Fútbol

## *Modelos no lineales para pronóstico*  - Pedro Martinez

---

## **<font color= #0077b6> Objetivos de la Práctica </font>**
1. Visualizar ejemplos de uso FFNN para predicción en series de tiempo.
2. Analizar el tamaño de test y train para una red neuronal FFNN.
2. Diseñar tamaño de ventanas deslizantes.

## **<font color= #0077b6> Descripción de la API </font>**

<img src="https://raw.githubusercontent.com/statsbomb/open-data/refs/heads/master/img/SB%20-%20Icon%20Lockup%20-%20Colour%20positive.png"
     align="CENTER"
     width="150"/>


Para esta práctica utilizaremos datos de StatsBomb, una de las empresas líderes en analítica deportiva a nivel global, la cual captura "eventos" (Event Data).

**¿Qué es un evento?**

Es cualquier acción que ocurre en el campo con una coordenada (x, y) y una marca de tiempo: pases, tiros, presiones, intercepciones o faltas.

**Volumen de datos:**

Un partido de fútbol promedio genera entre 3,000 y 4,000 eventos.

**Acceso:**

Utilizaremos la librería `statsbombpy`, que permite acceder a su Open Dataset.

## **<font color= #0077b6> Cálculo de momentum ofensivo </font>**

- En el fútbol, el gol es un evento de alta importancia pero muy baja frecuencia, lo que lo hace una serie de tiempo pésima para modelos de aprendizaje profundo.
- Para que la FFNN tenga suficiente información, construiremos una métrica de "Momentum Ofensivo".
- El momentum ofensivo es el volumen de pases completados en el último tercio de la cancha por cada minuto de juego.

**_Objetivo:_**
- Entrenar una (FFNN) para que aprenda la dinámica de intensidad del partido. El modelo deberá observar los últimos $w$ minutos (ventana deslizante) y predecir cuál será la intensidad ofensiva en el minuto siguiente.

In [3]:
from statsbombpy import sb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [4]:
# @title Visualizar partidos disponibles
# Ver todas las competiciones gratuitas disponibles
competiciones = sb.competitions()
print(competiciones[['competition_id', 'season_id', 'competition_name', 'season_name']])

# Ver los partidos del Mundial de Qatar 2022 (competition_id=43, season_id=106)
partidos_mundial = sb.matches(competition_id=43, season_id=106)

# Mostrar una tabla limpia con el ID del partido y los equipos
tabla_partidos = partidos_mundial[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']]
print(tabla_partidos.head(10))

KeyboardInterrupt: 

## **<font color= #0077b6> Selección de datos y filtrado </font>**

- Si analizamos los pases totales por minuto sin ningún filtro, nos enfrentamos a lo que en estadística se conoce como Ruido de Alta Frecuencia y Dispersión (Sparsity).
- Si le pasamos estos saltos violentos a una red neuronal, la función de pérdida (Loss Function) se volverá inestable.
- Al final, la red simplemente aprenderá a predecir el promedio histórico para minimizar su error, devolviendo una línea recta inútil.

In [5]:
# @title Graficamos serie de tiempo de solo pases por minuto
MATCH_ID = 3869321


eventos = sb.events(match_id=MATCH_ID)

# Filtrar pases completados en última parte de la cancha
pases = eventos[(eventos['type'] == 'Pass') & (eventos['pass_outcome'].isnull())]
pases['x_loc'] = pases['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
pases_ataque = pases[pases['x_loc'] >= 80]

# Agrupar contando los pases totales por minuto (sin separar por equipo)
pases_totales_minuto = pases_ataque.groupby('minute').size().reset_index(name='pases_totales')

# Crear el eje de tiempo continuo, rellenando los minutos vacíos con 0
minuto_max = eventos['minute'].max()
df_ts_crudo = pd.DataFrame({'minute': range(0, minuto_max + 1)})
df_ts_crudo = pd.merge(df_ts_crudo, pases_totales_minuto, on='minute', how='left').fillna(0)


fig = go.Figure()
fig.add_trace(go.Scatter(x=df_ts_crudo['minute'], y=df_ts_crudo['pases_totales'], name='Pases Totales',
                         line=dict(color='#00BD9D', width=3)))

fig.update_layout(title='Intensidad Agregada: Pases Totales por Minuto en Zona de Ataque',
                  xaxis_title='Minuto del Partido',
                  yaxis_title='Cantidad Total de Pases',
                  template='plotly_white')
fig.show()

KeyboardInterrupt: 

- La manera que usualmente se trata a estas señales erraticas es con suavizado.
- Para descubrir la verdadera "Intensidad" o presión táctica del partido, necesitamos crear una métrica con memoria. Usaremos una Media Móvil Simple (Simple Moving Average - SMA) de 5 minutos.Matemáticamente, para encontrar el Momentum en el minuto $t$ con una ventana $w=5$:$$SMA_t = \frac{p_t + p_{t-1} + p_{t-2} + p_{t-3} + p_{t-4}}{5}$$
- Tener una serie de esta manera, ayuda a tener un patrón suave y lógico que puede intentar aprender y predecir.

In [ ]:
# @title Graficamos serie de tiempo considerando el momentum ofensivo.
MATCH_ID = 3869321
eventos = sb.events(match_id=MATCH_ID)

# Identificar automáticamente a los equipos
equipos = eventos['team'].dropna().unique()
equipo_A, equipo_B = equipos[0], equipos[1]

# Asignar colores dinámicos para la gráfica
colores_equipos = {equipo_A: '#1f77b4', equipo_B: '#d62728'}

# Pases en el último tercio
pases = eventos[(eventos['type'] == 'Pass') & (eventos['pass_outcome'].isnull())]
pases['x_loc'] = pases['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
pases_ataque = pases[pases['x_loc'] >= 80]
# Conteo de pases enteros por minuto
pases_por_minuto = pases_ataque.groupby('minute').size().reset_index(name='pases')

# Crear serie de tiempo continua, rellenando minutos sin pases con 0
minuto_max = eventos['minute'].max()
df_ts = pd.DataFrame({'minute': range(0, minuto_max + 1)})
df_ts = pd.merge(df_ts, pases_por_minuto, on='minute', how='left').fillna(0)

# SUAVIZADO DE LA SERIE
# Transforma conteos enteros en un "Promedio de pases por minuto" suavizado
df_ts['Momentum'] = df_ts['pases'].rolling(window=5, min_periods=1).mean()

# Obtenemos los goles que hubo
goles = eventos[(eventos['type'] == 'Shot') & (eventos['shot_outcome'] == 'Goal')][['minute', 'team']]


fig = go.Figure()
fig.add_trace(go.Scatter(x=df_ts['minute'], y=df_ts['Momentum'],
                         mode='lines', name='Momentum Ofensivo',
                         line=dict(color='#2ca02c', width=3)))

# Agregar líneas verticales dinámicas para los goles
for index, row in goles.iterrows():
    minuto = row['minute']
    equipo = row['team']
    color_gol = colores_equipos.get(equipo, 'black')

    # Alternar posición para que no se encimen los textos
    posicion_texto = "top left" if index % 2 == 0 else "top right"

    fig.add_vline(x=minuto, line_width=2, line_dash="dash", line_color=color_gol,
                  annotation_text=f"⚽{minuto}'",
                  annotation_position=posicion_texto,
                  annotation_font=dict(color=color_gol, size=12))

fig.update_layout(title=f'Momentum Ofensivo: {equipo_A} vs {equipo_B}',
                  xaxis_title='Minuto del Partido',
                  yaxis_title='Promedio Móvil de Pases')
fig.show()

## **<font color= #0077b6> Preparación de dataset entrada para red neuronal </font>**

- Las Redes Neuronales necesitan datos escalados (generalmente entre 0 y 1) para que los gradientes converjan correctamente de una manera más rápida.
- Es importante realizar la separación de train y test ANTES de ejecutar el escalador.
-Si escalas todo junto, se tiene información de todo el partido. Se filtra información matemática del futuro hacia los datos del pasado.

In [ ]:
# @title Creamos ventanas y escalamiento

# Convertimos en arreglo 2d y eliminamos cualquier posible NA
serie_tiempo = df_ts['Momentum'].dropna().values.reshape(-1, 1)


# Seleccionamos train y test (80/20)
train_size = int(len(serie_tiempo) * 0.8)
train_data = serie_tiempo[:train_size]
test_data = serie_tiempo[train_size:]

# Iniciamos el escalador
scaler = MinMaxScaler(feature_range=(0, 1))

# Aplicamos el escalador SOLO a los datos de entrenamiento
train_scaled = scaler.fit_transform(train_data)

# Ahora escalador con parametros del train, pero para transformar los datos de test
test_scaled = scaler.transform(test_data)

# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

# SELECCIÓN DE TAMAÑO DE VENTANA
window_size = 5
X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

In [ ]:

# @title Visualización de las ventanas que serán entrada de la red neuronal

# Usamos 10 datos como ejemplo
n_viz = 10
X_viz = X_train[:n_viz]
y_viz = y_train[:n_viz].reshape(-1, 1)
matriz_viz = np.hstack((X_viz, y_viz))
# Redondeo
matriz_viz_redondeada = np.round(matriz_viz, 2)


etiquetas_x = [f't-{window_size - i}' for i in range(window_size)] + ['Objetivo (t)']
etiquetas_y = [f'Ventana {i}' for i in range(n_viz)]


fig = go.Figure(data=go.Heatmap(
    z=matriz_viz,
    x=etiquetas_x,
    y=etiquetas_y,
    text=matriz_viz_redondeada, # Agregamos los valores redondeados
    texttemplate="%{text}",     # Forzamos a que muestre el texto
    colorscale='Teal',
    showscale=True,
    xgap=2, ygap=2              # Separación entre celdas
))

# Línea divisoria
fig.add_vline(x=window_size - 0.5, line_width=4, line_color='red', line_dash='dot')

# 6. Diseño compacto
fig.update_layout(
    title='Entrada Red Neuronal Matriz de Entrenamiento Escalada',
    xaxis_title='Características de Entrada (Lags) | Salida Esperada',
    yaxis_title='Muestras',
    yaxis=dict(autorange="reversed"), # Invertir para leer de arriba hacia abajo
    width=800, height=500, # Hacerlo más pequeño y proporcionado
    template='plotly_white'
)

fig.show()

## **<font color= #0077b6> Creación de FFNN y predicciones </font>**


- Capa de Entrada: Tiene exactamente $w=5$ conexiones. Cada conexión recibe uno de los rezagos temporales ($t-5, \dots, t-1$).

- Capas Ocultas (Densas): Usaremos dos capas ocultas pequeñas de 16 y 8 neuronas con activación ReLU.

- Al tener pocos datos de un solo un partido, una red muy profunda memorizaría el ruido. Buscamos que extraiga patrones simples.

- Capa de Salida: Una sola neurona sin función de activación (activación lineal) ya que nuestro objetivo es predecir un número continuo, no queremos restringir la salida entre 0 y 1 como lo haría una Sigmoide.

- Evaluación: Evitamos el MAPE porque si en el minuto real hubo 0 pases, la división entre cero da error infinito. Usaremos MAE y RMSE que nos darán el margen de error en la misma unidad original: "pases por minuto".


In [ ]:
# @title Creación de red neuronal FFNN

# Iniciamos el modelo FFNN
model = Sequential([
    # Capa 1: input_dim DEBE coincidir con el window_size
    Dense(16, activation='relu', input_dim=window_size, name='Capa_Oculta_1'),

    # Capa 2: Extracción de características no lineales
    Dense(8, activation='relu', name='Capa_Oculta_2'),

    # Capa de Salida: 1 neurona Funcion de act LINEAL
    Dense(1, name='Salida_Pronostico')
])
model.summary()

# Usamos el optimizador Adam y MSE
optimizador = Adam(learning_rate=0.005)
model.compile(optimizer=optimizador, loss='mse')

history = model.fit(
    X_train, y_train,
    epochs=60,           # Cantidad de veces que verá el dataset completo
    batch_size=8,        # Actualiza los pesos cada 8 ventanas
    validation_data=(X_test, y_test), # Evaluamos en datos que no ha visto
    verbose=0            # 0 para no mostrar info
)



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Capa_Oculta_1 (Dense)           │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Oculta_2 (Dense)           │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Salida_Pronostico (Dense)       │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# @title Predicciones y gráficas


# El modelo genera predicciones (pero salen en escala de 0 a 1)
y_pred_scaled = model.predict(X_test, verbose=0)

# Invertimos el escalamiento para volver a "Pases por minuto"
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculamos errores
mae = mean_absolute_error(y_test_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))

print(f"MAE (Error Absoluto Medio): {mae:.2f} pases por minuto")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.2f} pases por minuto")


fig = go.Figure()


minutos_train = df_ts['minute'].iloc[window_size : train_size]
minutos_test = df_ts['minute'].iloc[train_size + window_size :]

# Entrenamiento
fig.add_trace(go.Scatter(x=minutos_train, y=scaler.inverse_transform(y_train.reshape(-1, 1)).flatten(),
                         mode='lines', name='Entrenamiento (Real)',
                         line=dict(color='#bdc3c7', width=1.5))) # Gris claro

# Test
fig.add_trace(go.Scatter(x=minutos_test, y=y_test_real.flatten(),
                         mode='lines+markers', name='Realidad (Test)',
                         line=dict(color='#2ca02c', width=2.5))) # Verde

# Pronóstico de la Red Neuronal
fig.add_trace(go.Scatter(x=minutos_test, y=y_pred.flatten(),
                         mode='lines+markers', name='Pronóstico FFNN',
                         line=dict(color='#e74c3c', dash='dot', width=2.5))) # Rojo punteado

# Línea divisoria
fig.add_vline(x=df_ts['minute'].iloc[train_size], line_width=2, line_dash="dash", line_color="black")

fig.update_layout(title='Forecast FFNN',
                  xaxis_title='Minuto del Partido',
                  yaxis_title='Promedio Móvil de Pases',
                  legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'))
fig.show()

MAE (Error Absoluto Medio): 0.41 pases por minuto
RMSE (Raíz del Error Cuadrático Medio): 0.51 pases por minuto
